# 6 - IP Clip Valley Bottom Polygons by RCA and Clean

Use this tool to clip a valley bottom polygon feature class by RCA polygons. Any instances of multiple polygons with the same RCA ID will be edited to keep only the largest of the polygons.

## Required Software:

- The code contained in this notebook is designed to be run within an ESRI ArcPro project. The script output is written to the default project geodatabase and therefore the user must open and run the notebook .IPYNB file within an ArcPro project.
- If any of the geoprocessing steps require an advanced license or any specific extensions, the script will check for these conditions before running.

## Required Inputs:

- A geodatabase containing RCA polygon and valley bottom polygon feature classes. The RCA feature class names must have an "RCA" prefix, and the the valley bottom polygon feature class names must have a "smoothed" suffix. These should be the output from the "IP 4 Build RCAs from Pre-Processed NetMap Reaches and DEM" and "IP 5 Resample IFSAR Mosaics and Create Valley Bottom Polygons" tools. Using these tools should result in an equal number of RCA and VB feature classes, with the appropriate feature class names.

- A field containing unique RCA IDs in the RCA feature classes.


## Geoprocessing Output:

- A feature class of cleaned valley bottom polygons clipped by RCAs, written to the same geodatabase containing the RCA and VB feature classes.

## Processing Steps:

1. List the feature classes in the user-provided geodatabase, and split the list into RCAs and valley bottoms using the prefixes and suffixes. 
2. Sort lists, then compare to make sure there are equal number of matching feature classes in each.
3. For each matching pair of RCAs and VBs, clip the VB polygons by RCA, using the RCA ID to group the output.
4. Sort the clipped results and save to new sorted feature classes.
5. For each sorted RCA VB feature class, run the cleaning process.
6. Delete intermediary layers.

### Code starts here:

#### Setup

Import modules and reset environments to default. This should set the ArcPro project geodatabase as the workspace/scratch environment, just in case it was set otherwise. Additionally, allow the addition of intermediary outputs to the ArcPro project map. (Some selection procedures do not work as expected if the feature classes are not loaded into the map.)

In [71]:
import arcpy
import pandas

arcpy.ResetEnvironments()
arcpy.env.addOutputsToMap = True

User provides filepaths to the geodatabase, and the field in the RCA feature class that contains the unique reach ID (usually "RCA_ID" in IP toolbox script outputs).

(These inputs are provided directly as text when running the code block from the notebook. Use the hashed out code block below if saving the notebook as a .PY file and creating a tool for use in the ArcPro GUI. In that case, the inputs will be provided as text parameters in "point-and-click" fashion when setting up the tool in the ArcPro GUI.)

In [1]:
gdb = "F:\\GIS\\IP\\Yukon_Upper_2.gdb"
rid = "RCA_ID"

In [73]:
#gdb = arcpy.GetParameterAsText(0)
#rid = arcpy.GetParameterAsText(1)

List the feature classes in the user-provided geodatabase, and split the list into RCAs and valley bottoms using the prefixes and suffixes.

In [2]:
arcpy.env.workspace = gdb

featureclasses = arcpy.ListFeatureClasses()

RCAs = []
VBs = []

for fc in featureclasses:
    if fc.startswith("RCA") == True:
        
        desc = arcpy.Describe(fc)
        fc_full = str(desc.path + "/" + fc)
        
        RCAs.append(fc_full)
        
    elif fc.endswith("smoothed") == True:
        
        desc = arcpy.Describe(fc)
        fc_full = str(desc.path + "/" + fc)
        
        VBs.append(fc_full)
    
    else:
        pass


print(RCAs)
print(VBs)

['F:\\GIS\\IP\\Yukon_Upper_2.gdb/RCA_19070402_HUC8', 'F:\\GIS\\IP\\Yukon_Upper_2.gdb/RCA_19070403_HUC8', 'F:\\GIS\\IP\\Yukon_Upper_2.gdb/RCA_19070502_HUC8', 'F:\\GIS\\IP\\Yukon_Upper_2.gdb/RCA_VB_19070402_clean', 'F:\\GIS\\IP\\Yukon_Upper_2.gdb/RCA_VB_19070403_clean', 'F:\\GIS\\IP\\Yukon_Upper_2.gdb/RCA_VB_19070502_clean']
[]


Check for an equal number of feature classes. If unequal, provide an error message and abort the rest of the script. 

If equal, use the HUC IDs top sort the valley bottom list to match the order of the RCA list.

In [75]:
if len(RCAs) == len(VBs):
    pass
else:
    print("Unequal number of RCA and Valley Bottom feature classes!  .... Exiting script...")
    sys.exit(0)




VBs_sorted = []

for r in RCAs:
    code = r.split("_")[-2]
    
    for v in VBs:

        v_code = v.split("_")[-4]
        
        if v_code == code:
            VBs_sorted.append(v)
        
        else:
            pass

VBs_sorted

['F:\\GIS\\IP\\Kuskokwim.gdb/reach_Kuskokwim_5km_1903050a_HUC8_VB600_smoothed', 'F:\\GIS\\IP\\Kuskokwim.gdb/reach_Kuskokwim_5km_1903050b_HUC8_VB600_smoothed', 'F:\\GIS\\IP\\Kuskokwim.gdb/reach_Kuskokwim_5km_19030401_HUC8_VB600_smoothed', 'F:\\GIS\\IP\\Kuskokwim.gdb/reach_Kuskokwim_5km_19030403_HUC8_VB600_smoothed', 'F:\\GIS\\IP\\Kuskokwim.gdb/reach_Kuskokwim_5km_19030404_HUC8_VB600_smoothed', 'F:\\GIS\\IP\\Kuskokwim.gdb/reach_Kuskokwim_5km_19030405_HUC8_VB600_smoothed', 'F:\\GIS\\IP\\Kuskokwim.gdb/reach_Kuskokwim_5km_19030406_HUC8_VB600_smoothed', 'F:\\GIS\\IP\\Kuskokwim.gdb/reach_Kuskokwim_5km_19030407_HUC8_VB600_smoothed', 'F:\\GIS\\IP\\Kuskokwim.gdb/reach_Kuskokwim_5km_19030501_HUC8_VB600_smoothed']

For each matching pair of RCAs and VBs, clip the VB polygons by RCA, using the RCA ID to group the output. Save the output names to a new list for cleaning.

In [76]:
RCA_VBs = []

for RCA, VB in zip(RCAs, VBs_sorted):
    
    outname = str("RCA_VB_" + str(RCA.split("_")[-2]))
    
    arcpy.analysis.Intersect([VB, RCA], outname)
    
    RCA_VBs.append(outname)

Using the user-provided unqiue RCA ID field and the default area field, sort the features in each RCA VB feature class. Save each sorted output to a new list.

In [77]:
RCA_VBs_sorted = []

for rca_vb in RCA_VBs:

    sort_fields = [[rid, "ASCENDING"], ["Shape_Area", "ASCENDING"]]
    
    rca_vb_sorted = str(rca_vb + "_sorted")

    arcpy.management.Sort(rca_vb, rca_vb_sorted, sort_fields)
    
    RCA_VBs_sorted.append(rca_vb_sorted)

For each sorted RCA VB feature class, run the cleaning process.

This process checks for duplicate RCA IDs in the RCA feature class (ie, RCAs with multiple polygons), lists the duplicates along with their areas and object IDs. Any duplicate polygons that are not the polygon with largest area are listed and used to make a selection from the sorted RCA VB feature class. The selection is reversed and the result is copied to a new "clean" feature class, which will have only one RCA VB polygon per RCA ID.

In [78]:
for RCA_VB_s in RCA_VBs_sorted:
    
    # list all RCA VB ids...
    with arcpy.da.SearchCursor(RCA_VB_s, [rid]) as rows:
        values = [r[0] for r in rows]


    # create an empty data frame to hold duplicates
    df = pandas.DataFrame(columns = ["OBJECTID", rid, "Shape_Area"])

    
    # loop thru the valley bottom features, checking if each feature's RCA id is a duplicate
    # if so, add the OID, RCA id, and shape area to the empty dataframe
    with arcpy.da.SearchCursor(RCA_VB_s, ["OBJECTID", rid, "Shape_Area"]) as rows:
        for row in rows:
            if values.count(row[1]) > 1:

                oid = row[0]
                rca = row[1]
                area = row[2]

                i = len(df)
                df.loc[i+1] = [oid, rca, area]

            else:
                pass


    # convert back to integer
    df = df.astype({"OBJECTID": int, rid: int})


    # since the RCA VB features (and their OIDs) were already sorted by size when read into the dataframe, 
    # the dataframe should preserve that size order. Keeping the "last" row of duplicates should only
    # keep the largest sized feature and also keep the OID column

    df_ = df.drop_duplicates(subset=rid, keep="last")


    # compare the full list of duplicate OIDS to the list of largest feature OIDs... 
    # make a new list with just the smaller features
    all_oids = df["OBJECTID"].values.tolist()
    big_oids = df_["OBJECTID"].values.tolist()

    small_oids = [o for o in all_oids if o not in big_oids]


    print(len(all_oids))
    print(len(big_oids))
    print(len(small_oids))


    # loop thru the list of small OIDs, building a where clause and adding each to the selection
    # then delete the selected features... the layer MUST be in the map display for this to work!
    # add an if statement to check if anything is selected... this avoids 
    # deleting the entire feature class if selection was unsuccessful

    for oid in small_oids:
        where = str("OBJECTID = " + str(oid))
        arcpy.management.SelectLayerByAttribute(RCA_VB_s, "ADD_TO_SELECTION", where)

    arcpy.SelectLayerByAttribute_management(RCA_VB_s, "SWITCH_SELECTION")

    pre = str(RCA_VB_s.split("_sorted")[0])
    outclean = str(pre + "_clean")
    
    arcpy.management.CopyFeatures(RCA_VB_s, outclean)

    #arcpy.management.Delete("sorted_vb")

2888
644
2244
2016
534
1482
1034
274
760
296
101
195
997
333
664
2250
625
1625
432
119
313
357
107
250
1241
264
977


Delete intermediary layers.

In [80]:
arcpy.management.Delete(RCA_VBs)
arcpy.management.Delete(RCA_VBs_sorted)

<Result 'true'>